In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

print("Libraries loaded successfully.")

# Find project root
project_root = Path.cwd()

# If notebook is running from the cleaning folder,
# move one level up to DSS_Dashboard
if project_root.name == "cleaning":
    project_root = project_root.parent

input_file = project_root / "data" / "ejis_raw.csv"

print("Project root:", project_root)
print("Input file:", input_file)
print("File exists:", input_file.exists())

df = pd.read_csv(input_file)

print("\nRows:", len(df))
print("Columns:", len(df.columns))

df.head()

Libraries loaded successfully.
Project root: /Users/keerthisagi/Documents/DSS_Dashboard
Input file: /Users/keerthisagi/Documents/DSS_Dashboard/data/ejis_raw.csv
File exists: True

Rows: 757
Columns: 15


,DOI,Title,Year,Date,Author,Institution,Country,Citation count,Open access,Abstract,Keyword,Topic,Subfield,Field,Domain
0,https://doi.org/10.1057/ejis.2011.51,Using grounded theory as a method for rigorous...,2011,2011-11-29,Joost F. Wolfswinkel|Elfi Furtmueller|Celeste ...,University of Twente,NL|NL|NL,1165,NOT Open Access,NaN,grounded theory|research rigor|systematic revi...,Qualitative Research Methods and Applications,Education,Social Sciences,Social Sciences
1,https://doi.org/10.1057/ejis.2012.26,A method for taxonomy development and its appl...,2012,2012-06-19,Robert C. Nickerson|Upkar Varshney|Jan Muntermann,San Francisco State University|Georgia State U...,US|US|DE,1151,Open Access,A fundamental problem in many disciplines is t...,taxonomy development|object classification|inf...,Information Systems Theories and Implementation,Sociology and Political Science,Social Sciences,Social Sciences
2,https://doi.org/10.1057/ejis.2014.36,FEDS: a Framework for Evaluation in Design Sci...,2014,2014-11-11,John R. Venable|Jan Pries‐Heje|Richard Baskerv...,Curtin University|Roskilde University|Georgia ...,AU|DK|AU|US,1069,Open Access,Evaluation of design artefacts and design theo...,design science research|evaluation framework|a...,Information Systems Theories and Implementation,Sociology and Political Science,Social Sciences,Social Sciences
3,https://doi.org/10.1057/ejis.2010.21,Developing a unified framework of the business...,2010,2010-05-11,Mutaz M. Al‐Debei|David Avison,Brunel University of London|École Supérieure d...,GB|FR,880,NOT Open Access,NaN,business models|business model concept|concept...,Business Strategies and Innovation,Strategy and Management,"Business, Management and Accounting",Social Sciences
4,https://doi.org/10.1080/0960085x.2021.1927212,"Algorithmic bias: review, synthesis, and futur...",2021,2021-06-06,Nima Kordzadeh|Maryam Ghasemaghaei,Worcester Polytechnic Institute|McMaster Unive...,US|CA,761,NOT Open Access,NaN,algorithmic bias|data-driven decision making|p...,Ethics and Social Impacts of AI,Safety Research,Social Sciences,Social Sciences


In [2]:
print("EJIS RAW COLUMNS")
print("=" * 50)

for i, column in enumerate(df.columns, start=1):
    print(f"{i:2}. {column}")

EJIS RAW COLUMNS
 1. DOI
 2. Title
 3. Year
 4. Date
 5. Author
 6. Institution
 7. Country
 8. Citation count
 9. Open access
10. Abstract
11. Keyword
12. Topic
13. Subfield
14. Field
15. Domain


In [3]:
summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique()
})

summary

,Data Type,Missing,Missing %,Unique Values
DOI,object,1,0.13,756
Title,object,0,0.00,754
Year,int64,0,0.00,17
Date,object,0,0.00,550
Author,object,2,0.26,711
Institution,object,30,3.96,657
Country,object,21,2.77,409
Citation count,int64,0,0.00,203
Open access,object,0,0.00,2
Abstract,object,422,55.75,335


In [4]:
# ==========================================
# STEP 1: CHECK DUPLICATES
# ==========================================

print("Total rows:", len(df))

print(
    "Exact duplicate rows:",
    df.duplicated().sum()
)

print(
    "Missing DOIs:",
    df["DOI"].isna().sum()
)

print(
    "Duplicate DOIs:",
    df["DOI"].dropna().duplicated().sum()
)

print(
    "Duplicate titles:",
    df["Title"].duplicated().sum()
)


# Inspect all repeated titles
duplicate_titles = df[
    df["Title"].duplicated(keep=False)
].sort_values(
    by=["Title", "Year"]
)

duplicate_titles[
    [
        "DOI",
        "Title",
        "Year",
        "Date"
    ]
]

Total rows: 757
Exact duplicate rows: 0
Missing DOIs: 1
Duplicate DOIs: 0
Duplicate titles: 3


,DOI,Title,Year,Date
735,https://doi.org/10.1080/0960085x.2026.2630408,Correction,2026,2026-02-18
753,https://doi.org/10.1080/0960085x.2026.2718654,Correction,2026,2026-08-17
728,https://doi.org/10.1057/s41303-017-0037-2,Thanks to Reviewers,2017,2017-01-01
725,https://doi.org/10.1080/0960085x.2018.1425658,Thanks to Reviewers,2018,2018-01-02
727,https://doi.org/10.1080/0960085x.2019.1565656,Thanks to Reviewers,2019,2019-01-02


In [5]:
# ==========================================
# STEP 2: BASIC TEXT AND DOI CLEANING
# ==========================================

def clean_text(value):
    if pd.isna(value):
        return value

    value = str(value)

    # Remove line breaks and tabs
    value = value.replace("\n", " ")
    value = value.replace("\r", " ")
    value = value.replace("\t", " ")

    # Remove excessive spaces
    value = re.sub(r"\s+", " ", value)

    value = value.strip()

    if not value:
        return pd.NA

    return value


def clean_doi(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    value = value.replace(
        "https://doi.org/",
        ""
    )

    value = value.replace(
        "http://doi.org/",
        ""
    )

    if not value:
        return pd.NA

    return value


# Clean article titles
df["Title"] = df["Title"].apply(clean_text)

# Clean DOI
df["DOI"] = df["DOI"].apply(clean_doi)


# Validate DOI cleaning
print("Missing DOI:", df["DOI"].isna().sum())

print(
    "Duplicate DOI:",
    df["DOI"].dropna().duplicated().sum()
)

print("\nSample cleaned DOIs:")
print(
    df["DOI"]
    .dropna()
    .head(10)
    .to_string(index=False)
)

Missing DOI: 1
Duplicate DOI: 0

Sample cleaned DOIs:
         10.1057/ejis.2011.51
         10.1057/ejis.2012.26
         10.1057/ejis.2014.36
         10.1057/ejis.2010.21
10.1080/0960085x.2021.1927212
10.1080/0960085x.2020.1800417
          10.1057/ejis.2012.1
          10.1057/ejis.2014.3
          10.1057/ejis.2013.4
          10.1057/ejis.2014.7


In [6]:
# ==========================================
# STEP 3: CLEAN AND VALIDATE DATE
# ==========================================

# Convert Date to datetime
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

# Ensure Year is numeric
df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
).astype("Int64")


print("Date type:", df["Date"].dtype)
print("Year type:", df["Year"].dtype)

print(
    "\nInvalid dates:",
    df["Date"].isna().sum()
)

print(
    "Missing years:",
    df["Year"].isna().sum()
)

print(
    "\nEarliest date:",
    df["Date"].min()
)

print(
    "Latest date:",
    df["Date"].max()
)

print(
    "Year range:",
    df["Year"].min(),
    "-",
    df["Year"].max()
)


# Check whether Year agrees with Date
year_mismatch = df[
    df["Date"].notna()
    & df["Year"].notna()
    & (df["Year"] != df["Date"].dt.year)
]

print(
    "\nYear/date mismatches:",
    len(year_mismatch)
)

year_mismatch[
    [
        "DOI",
        "Title",
        "Year",
        "Date"
    ]
]

Date type: datetime64[ns]
Year type: Int64

Invalid dates: 0
Missing years: 0

Earliest date: 2010-01-19 00:00:00
Latest date: 2026-09-16 00:00:00
Year range: 2010 - 2026

Year/date mismatches: 0


,DOI,Title,Year,Date


In [7]:
# ==========================================
# STEP 4: CLEAN MULTI-VALUE COLUMNS
# ==========================================

def clean_multi_value(value):
    """
    Cleans values separated by |
    while keeping one article per row.
    """

    if pd.isna(value):
        return pd.NA

    # Split values
    values = str(value).split("|")

    cleaned = []
    seen = set()

    for item in values:

        # Clean spaces
        item = item.strip()

        # Ignore empty values
        if not item:
            continue

        # Remove duplicate values
        # Case-insensitive comparison
        key = item.lower()

        if key not in seen:
            cleaned.append(item)
            seen.add(key)

    if not cleaned:
        return pd.NA

    return " | ".join(cleaned)


multi_columns = [
    "Author",
    "Institution",
    "Country",
    "Keyword",
    "Topic",
    "Subfield",
    "Field",
    "Domain"
]

for column in multi_columns:
    df[column] = df[column].apply(clean_multi_value)

print("Multi-value columns cleaned.")

df[
    [
        "Author",
        "Institution",
        "Country",
        "Keyword",
        "Topic"
    ]
].head(10)

Multi-value columns cleaned.


,Author,Institution,Country,Keyword,Topic
0,Joost F. Wolfswinkel | Elfi Furtmueller | Cele...,University of Twente,NL,grounded theory | research rigor | systematic ...,Qualitative Research Methods and Applications
1,Robert C. Nickerson | Upkar Varshney | Jan Mun...,San Francisco State University | Georgia State...,US | DE,taxonomy development | object classification |...,Information Systems Theories and Implementation
2,John R. Venable | Jan Pries‐Heje | Richard Bas...,Curtin University | Roskilde University | Geor...,AU | DK | US,design science research | evaluation framework...,Information Systems Theories and Implementation
3,Mutaz M. Al‐Debei | David Avison,Brunel University of London | École Supérieure...,GB | FR,business models | business model concept | con...,Business Strategies and Innovation
4,Nima Kordzadeh | Maryam Ghasemaghaei,Worcester Polytechnic Institute | McMaster Uni...,US | CA,algorithmic bias | data-driven decision making...,Ethics and Social Impacts of AI
5,Lena Waizenegger | Brad McKenna | Wenjie Cai |...,Auckland University of Technology | University...,NZ | GB,affordance theory | team collaboration | COVID...,Work-Family Balance Challenges
6,Ofir Turel | Alexander Serenko,"California State University, Fullerton | Lakeh...",US | CA,enjoyment | social networking websites | techn...,Technology Adoption and User Behaviour
7,Christian Maier | Sven Laumer | Andreas Eckhar...,University of Bamberg | Goethe University Fran...,DE,social overload | social networking sites | so...,Impact of Technology on Adolescents
8,Yang Chen | Yi Wang | Saggi Nevo | Jiafei Jin ...,Southwestern University of Finance and Economi...,CN | US | HK,IT capabilities | organizational performance |...,Information Technology Governance and Strategy
9,Frantz Rowe,SKEMA Business School | Laboratoire d'Economie...,FR,information systems research | diversity in re...,Meta-analysis and systematic reviews


In [8]:
# ==========================================
# STEP 5: CREATE COUNT COLUMNS
# ==========================================

def count_multi_values(value):

    if pd.isna(value):
        return 0

    return len(
        [
            item
            for item in str(value).split("|")
            if item.strip()
        ]
    )


df["Author count"] = (
    df["Author"].apply(count_multi_values)
)

df["Institution count"] = (
    df["Institution"].apply(count_multi_values)
)

df["Country count"] = (
    df["Country"].apply(count_multi_values)
)

df["Keyword count"] = (
    df["Keyword"].apply(count_multi_values)
)

df["Topic count"] = (
    df["Topic"].apply(count_multi_values)
)


print("Count columns created.")

df[
    [
        "Title",
        "Author count",
        "Institution count",
        "Country count",
        "Keyword count",
        "Topic count"
    ]
].head(10)

Count columns created.


,Title,Author count,Institution count,Country count,Keyword count,Topic count
0,Using grounded theory as a method for rigorous...,3,1,1,4,1
1,A method for taxonomy development and its appl...,3,3,2,5,1
2,FEDS: a Framework for Evaluation in Design Sci...,3,3,3,9,1
3,Developing a unified framework of the business...,2,2,2,10,1
4,"Algorithmic bias: review, synthesis, and futur...",2,2,2,9,1
5,An affordance perspective of team collaboratio...,4,3,2,8,1
6,The benefits and dangers of enjoyment with soc...,2,2,2,8,1
7,Giving too much social support: social overloa...,4,2,1,9,1
8,IT capability and organizational performance: ...,6,5,3,10,1
9,"What literature review is not: diversity, boun...",1,4,1,6,1


In [9]:
# ==========================================
# STEP 6: COLLABORATION INDICATORS
# ==========================================

df["International collaboration"] = (
    df["Country count"] > 1
)

df["Multi-institution collaboration"] = (
    df["Institution count"] > 1
)


print("Collaboration indicators created.")

print(
    "\nInternational collaboration:"
)
print(
    df["International collaboration"].value_counts()
)

print(
    "\nMulti-institution collaboration:"
)
print(
    df["Multi-institution collaboration"].value_counts()
)


Collaboration indicators created.

International collaboration:
International collaboration
False    420
True     337
Name: count, dtype: int64

Multi-institution collaboration:
Multi-institution collaboration
True     548
False    209
Name: count, dtype: int64


In [10]:
# ==========================================
# STEP 7A: INSPECT OPEN ACCESS VALUES
# ==========================================

print("Open Access values before cleaning:")
print()

print(
    df["Open access"].value_counts(
        dropna=False
    )
)

Open Access values before cleaning:

Open access
NOT Open Access    420
Open Access        337
Name: count, dtype: int64


In [11]:
# ==========================================
# STEP 7B: CLEAN OPEN ACCESS
# ==========================================

def clean_open_access(value):

    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().lower()

    if value in [
        "true",
        "yes",
        "1",
        "open",
        "open access"
    ]:
        return "Open Access"

    if value in [
        "false",
        "no",
        "0",
        "closed",
        "closed access",
        "not open access"
    ]:
        return "Closed Access"

    return "Unknown"


df["Open access"] = (
    df["Open access"]
    .apply(clean_open_access)
)


print("Open Access values after cleaning:")
print()

print(
    df["Open access"].value_counts(
        dropna=False
    )
)

Open Access values after cleaning:

Open access
Closed Access    420
Open Access      337
Name: count, dtype: int64


In [12]:
# ==========================================
# STEP 8: ABSTRACT AVAILABILITY
# ==========================================

df["Has abstract"] = (
    df["Abstract"].notna()
)

print("Abstract availability:")
print(
    df["Has abstract"].value_counts()
)

Abstract availability:
Has abstract
False    422
True     335
Name: count, dtype: int64


In [13]:
# ==========================================
# STEP 9: DATA AVAILABILITY FLAGS
# ==========================================

df["Has author"] = df["Author"].notna()

df["Has institution"] = (
    df["Institution"].notna()
)

df["Has country"] = (
    df["Country"].notna()
)

df["Has keyword"] = (
    df["Keyword"].notna()
)

df["Has topic"] = (
    df["Topic"].notna()
)


availability = pd.DataFrame({
    "Available": [
        df["Has author"].sum(),
        df["Has institution"].sum(),
        df["Has country"].sum(),
        df["Has abstract"].sum(),
        df["Has keyword"].sum(),
        df["Has topic"].sum()
    ]
}, index=[
    "Author",
    "Institution",
    "Country",
    "Abstract",
    "Keyword",
    "Topic"
])

availability["Missing"] = (
    len(df) - availability["Available"]
)

availability["Coverage %"] = (
    availability["Available"]
    / len(df)
    * 100
).round(2)

availability

,Available,Missing,Coverage %
Author,755,2,99.74
Institution,727,30,96.04
Country,727,30,96.04
Abstract,335,422,44.25
Keyword,753,4,99.47
Topic,752,5,99.34


In [14]:
# ==========================================
# STEP 10: VALIDATE CITATIONS
# ==========================================

df["Citation count"] = pd.to_numeric(
    df["Citation count"],
    errors="coerce"
).fillna(0).astype(int)


print(
    "Minimum citations:",
    df["Citation count"].min()
)

print(
    "Maximum citations:",
    df["Citation count"].max()
)

print(
    "Total citations:",
    df["Citation count"].sum()
)

print(
    "Average citations:",
    round(df["Citation count"].mean(), 2)
)

print(
    "Negative citation counts:",
    (df["Citation count"] < 0).sum()
)

Minimum citations: 0
Maximum citations: 1165
Total citations: 53243
Average citations: 70.33
Negative citation counts: 0


In [15]:
# ==========================================
# STEP 11: ADD JOURNAL
# ==========================================

df["Journal"] = "European Journal of Information Systems"

# Move Journal to the first column
column_order = [
    "Journal"
] + [
    column
    for column in df.columns
    if column != "Journal"
]

df = df[column_order]


print("Journal column added.")
print()

print(
    df["Journal"].value_counts()
)

print(
    "\nTotal columns:",
    len(df.columns)
)

print(
    "\nFirst 5 columns:",
    df.columns[:5].tolist()
)

Journal column added.

Journal
European Journal of Information Systems    757
Name: count, dtype: int64

Total columns: 29

First 5 columns: ['Journal', 'DOI', 'Title', 'Year', 'Date']


In [16]:
# ==========================================
# STEP 12: FINAL VALIDATION
# ==========================================

print("=" * 60)
print("FINAL EJIS DATASET VALIDATION")
print("=" * 60)

print("\nDATASET SIZE")
print("Rows:", len(df))
print("Columns:", len(df.columns))


print("\nIDENTIFIERS")
print(
    "Duplicate DOIs:",
    df["DOI"].dropna().duplicated().sum()
)

print(
    "Missing DOI:",
    df["DOI"].isna().sum()
)

print(
    "Missing Title:",
    df["Title"].isna().sum()
)


print("\nDATE")
print(
    "Missing Year:",
    df["Year"].isna().sum()
)

print(
    "Invalid Date:",
    df["Date"].isna().sum()
)

print(
    "Year range:",
    df["Year"].min(),
    "-",
    df["Year"].max()
)

year_mismatch = df[
    df["Date"].notna()
    & df["Year"].notna()
    & (df["Year"] != df["Date"].dt.year)
]

print(
    "Year/date mismatches:",
    len(year_mismatch)
)


print("\nCITATIONS")
print(
    "Minimum:",
    df["Citation count"].min()
)

print(
    "Maximum:",
    df["Citation count"].max()
)

print(
    "Total:",
    df["Citation count"].sum()
)

print(
    "Average:",
    round(df["Citation count"].mean(), 2)
)

print(
    "Negative values:",
    (df["Citation count"] < 0).sum()
)


print("\nOPEN ACCESS")
print(
    df["Open access"].value_counts(
        dropna=False
    )
)


print("\nMETADATA COVERAGE")
print(availability)


print("\nJOURNAL")
print(
    df["Journal"].value_counts()
)


print("\nCOLUMN ORDER")
for i, column in enumerate(
    df.columns,
    start=1
):
    print(f"{i:2}. {column}")

FINAL EJIS DATASET VALIDATION

DATASET SIZE
Rows: 757
Columns: 29

IDENTIFIERS
Duplicate DOIs: 0
Missing DOI: 1
Missing Title: 0

DATE
Missing Year: 0
Invalid Date: 0
Year range: 2010 - 2026
Year/date mismatches: 0

CITATIONS
Minimum: 0
Maximum: 1165
Total: 53243
Average: 70.33
Negative values: 0

OPEN ACCESS
Open access
Closed Access    420
Open Access      337
Name: count, dtype: int64

METADATA COVERAGE
             Available  Missing  Coverage %
Author             755        2       99.74
Institution        727       30       96.04
Country            727       30       96.04
Abstract           335      422       44.25
Keyword            753        4       99.47
Topic              752        5       99.34

JOURNAL
Journal
European Journal of Information Systems    757
Name: count, dtype: int64

COLUMN ORDER
 1. Journal
 2. DOI
 3. Title
 4. Year
 5. Date
 6. Author
 7. Institution
 8. Country
 9. Citation count
10. Open access
11. Abstract
12. Keyword
13. Topic
14. Subfield
15. Fiel